# DDM Weibull Collapsing Bound Analysis
## nxx1, seed=42, gain=1.0
Model: v ~ 1 + coherence (Weibull collapsing bound)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
%matplotlib inline

In [ ]:
import xarray as xr
idata = xr.open_datatree('/Users/aliciasmacbookair/Desktop/rnn_hssm_output/ddm_weibull_nxx1_s42_g1.0')
print(idata)

## Model Summary

In [ ]:
# Parameter summary
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
summary_data = []
for param in params:
    data = posterior[param].values.flatten()
    summary_data.append({
        'parameter': param,
        'mean': data.mean(),
        'sd': data.std(),
        'hdi_3%': np.percentile(data, 3),
        'hdi_97%': np.percentile(data, 97),
    })
display(pd.DataFrame(summary_data).set_index('parameter').round(3))
print("\nNote: p_outlier fixed at 0.05 (default lapse probability)")

## Traces

In [ ]:
# Plot traces manually
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
fig, axes = plt.subplots(len(params), 2, figsize=(12, 3*len(params)))
for i, param in enumerate(params):
    data = posterior[param].values  # shape: (chains, draws)
    # KDE plot
    for chain in data:
        axes[i, 0].hist(chain, bins=30, alpha=0.5, density=True)
    axes[i, 0].set_title(param)
    axes[i, 0].set_xlabel('value')
    # Trace plot
    for chain in data:
        axes[i, 1].plot(chain, alpha=0.7)
    axes[i, 1].set_title(f'{param} trace')
plt.tight_layout()
plt.show()

## Posterior Distributions

In [ ]:
# Plot posterior distributions
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
fig, axes = plt.subplots(1, len(params), figsize=(15, 3))
for i, param in enumerate(params):
    data = posterior[param].values.flatten()
    axes[i].hist(data, bins=50, density=True, alpha=0.7, color='steelblue')
    axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.3f}')
    axes[i].set_title(param)
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Effect of Coherence on Drift Rate

In [ ]:
import xarray as xr
posterior = idata.posterior
v_intercept = float(posterior['v_Intercept'].mean())
v_coherence = float(posterior['v_coherence'].mean())

coherence_vals = np.linspace(0, 0.15, 100)
drift_rate = v_intercept + coherence_vals * v_coherence

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(coherence_vals, drift_rate, color='blue')
ax.set_xlabel('Coherence (|coh|)')
ax.set_ylabel('Drift rate (v)')
ax.set_title('Effect of coherence on drift rate\nnxx1, seed=42, gain=1.0')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"v_Intercept: {v_intercept:.3f}")
print(f"v_coherence: {v_coherence:.3f}")

## Posterior Predictive Check

## Posterior Predictive Check

In [ ]:
# PPC — RT distribution split by correct vs incorrect
if 'posterior_predictive' in idata:
    pp = idata['posterior_predictive'].to_dataset()
    obs = idata['observed_data'].to_dataset()
    
    # Extract observed RT and response
    obs_rt = obs['rt,response'].values[:, 0]
    obs_resp = obs['rt,response'].values[:, 1]
    
    # Extract predicted RT and response (mean across posterior samples)
    pred_rt_all = pp['rt,response'].values[..., 0]
    pred_resp_all = pp['rt,response'].values[..., 1]
    
    # Flatten posterior samples
    pred_rt = pred_rt_all.flatten()
    pred_resp = pred_resp_all.flatten()
    
    # Split by correct vs incorrect
    obs_correct_rt = obs_rt[obs_resp == 1.0]
    obs_error_rt = obs_rt[obs_resp == -1.0]
    pred_correct_rt = pred_rt[pred_resp == 1.0]
    pred_error_rt = pred_rt[pred_resp == -1.0]
    
    # Zoom in to reasonable RT range
    rt_max = np.percentile(obs_rt, 99)
    bins = np.linspace(0, rt_max, 50)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Correct trials
    axes[0].hist(obs_correct_rt, bins=bins, density=True, alpha=0.6, 
                 color='blue', label='observed')
    axes[0].hist(pred_correct_rt[pred_correct_rt <= rt_max], bins=bins, 
                 density=True, alpha=0.4, color='red', label='predicted')
    axes[0].set_xlabel('RT (s)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Correct Trials')
    axes[0].legend()
    axes[0].set_xlim(0, rt_max)
    
    # Error trials
    axes[1].hist(obs_error_rt, bins=bins, density=True, alpha=0.6, 
                 color='blue', label='observed')
    axes[1].hist(pred_error_rt[pred_error_rt <= rt_max], bins=bins, 
                 density=True, alpha=0.4, color='red', label='predicted')
    axes[1].set_xlabel('RT (s)')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Error Trials')
    axes[1].legend()
    axes[1].set_xlim(0, rt_max)
    
    plt.suptitle('Posterior Predictive Check — RT distributions\nnxx1, seed=42, gain=1.0 (weibull bound)')
    plt.tight_layout()
    plt.show()
    
    print(f"Observed: {len(obs_correct_rt)} correct, {len(obs_error_rt)} errors")
    print(f"Predicted accuracy: {(pred_resp==1.0).mean():.3f}")
else:
    print("No posterior predictive samples found.")

## Quantile Probability Plot

In [ ]:
# Quantile probability plot
if 'posterior_predictive' in idata:
    obs = idata['observed_data'].to_dataset()
    pp = idata['posterior_predictive'].to_dataset()
    
    obs_rt = obs['rt,response'].values[:, 0]
    obs_resp = obs['rt,response'].values[:, 1]
    pred_rt_all = pp['rt,response'].values[..., 0]
    pred_resp_all = pp['rt,response'].values[..., 1]
    
    quantiles = [0.1, 0.3, 0.5, 0.7, 0.9]
    
    # Observed quantiles split by response
    obs_correct_rt = obs_rt[obs_resp == 1]
    obs_error_rt = obs_rt[obs_resp == -1]
    obs_q_correct = np.quantile(obs_correct_rt, quantiles) if len(obs_correct_rt) > 0 else np.full(len(quantiles), np.nan)
    obs_q_error = np.quantile(obs_error_rt, quantiles) if len(obs_error_rt) > 0 else np.full(len(quantiles), np.nan)
    
    # Predicted quantiles (mean across posterior samples)
    pred_rt_flat = pred_rt_all.reshape(-1, pred_rt_all.shape[-1])
    pred_resp_flat = pred_resp_all.reshape(-1, pred_resp_all.shape[-1])
    
    pred_q_correct = []
    pred_q_error = []
    for i in range(pred_rt_flat.shape[0]):
        rt = pred_rt_flat[i]
        resp = pred_resp_flat[i]
        correct_rt = rt[resp == 1]
        error_rt = rt[resp == -1]
        pred_q_correct.append(np.quantile(correct_rt, quantiles) if len(correct_rt) > 0 else np.full(len(quantiles), np.nan))
        pred_q_error.append(np.quantile(error_rt, quantiles) if len(error_rt) > 0 else np.full(len(quantiles), np.nan))
    
    pred_q_correct = np.nanmean(pred_q_correct, axis=0)
    pred_q_error = np.nanmean(pred_q_error, axis=0)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(obs_q_correct, pred_q_correct, 'bo-', label='correct')
    ax.plot(obs_q_error, pred_q_error, 'rs-', label='error')
    ax.plot([0, max(obs_rt)], [0, max(obs_rt)], 'k--', alpha=0.4, label='identity')
    for i, q in enumerate(quantiles):
        ax.annotate(f'{int(q*100)}%', (obs_q_correct[i], pred_q_correct[i]), 
                   textcoords='offset points', xytext=(5,5), fontsize=8)
    ax.set_xlabel('Observed RT quantiles (s)')
    ax.set_ylabel('Predicted RT quantiles (s)')
    ax.set_title('Quantile Probability Plot\nnxx1, seed=42, gain=1.0 (weibull bound)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No posterior predictive samples found in idata.")

In [ ]:
# Check if posterior predictive is available
if 'posterior_predictive' in idata:
    pp = idata['posterior_predictive'].to_dataset()
    var_names = list(pp.data_vars)
    fig, axes = plt.subplots(1, len(var_names), figsize=(5*len(var_names), 4))
    if len(var_names) == 1:
        axes = [axes]
    for ax, var in zip(axes, var_names):
        data = pp[var].values.flatten()
        ax.hist(data, bins=50, density=True, alpha=0.7, color='steelblue')
        ax.set_title(f'Posterior predictive: {var}')
    plt.tight_layout()
    plt.show()
else:
    print("No posterior predictive samples found. Run model.sample_posterior_predictive() to generate.")